In [10]:
from torchvision.datasets import MNIST
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, Dataset
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [11]:
img_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
    transforms.Lambda(lambda x: x.to(device))
])

In [12]:
trn_ds = MNIST(root='', train=True, download=True, transform=img_transform)
val_ds = MNIST(root='', train=False, download=True, transform=img_transform)

In [13]:
trn_dl = DataLoader(trn_ds, batch_size=1024, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=1024, shuffle=False)

In [14]:
class AutoEncoder(nn.Module):
    def __init__(self, latent_dim):
        super(AutoEncoder, self).__init__()
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(
            nn.Linear(28*28, 128),
            nn.ReLU(True),
            nn.Linear(128, 64),
            nn.ReLU(True),
            nn.Linear(64, 12),
            nn.ReLU(True),
            nn.Linear(12, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 12),nn.ReLU(True),
            nn.Linear(12, 64),nn.ReLU(True),
            nn.Linear(64, 128),nn.ReLU(True),
            nn.Linear(128, 28*28),nn.Tanh(),
        )
        
    def forward(self, x):
        z = self.encoder(x)
        y = self.decoder(z)
        return y

In [15]:
from torchsummary import summary
model = AutoEncoder(latent_dim=3).to(device)
# print(summary(model, (1, 28*28)))

In [16]:
def train(model, trn_dl, criterion, optimizer):
    print("Training...")
    model.train()
    running_loss = 0.0
    for ix, (imgs, target) in enumerate(trn_dl):
        imgs = imgs.view(imgs.size(0), -1).to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, imgs)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
    epoch_loss = running_loss / len(trn_dl.dataset)
    return epoch_loss

def val(model, val_dl, criterion):
    model.eval()
    running_loss = 0.0
    for ix, (imgs, target) in enumerate(val_dl):
        imgs = imgs.view(imgs.size(0), -1).to(device)
        outputs = model(imgs)
        loss = criterion(outputs, imgs)
        running_loss += loss.item() * imgs.size(0)
    epoch_loss = running_loss / len(val_dl.dataset)
    return epoch_loss


In [ ]:
num_epochs = 20
train_losses = []
val_losses = []

for epoch in range(num_epochs):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    
    train_loss = train(model, trn_dl, criterion, optimizer)
    val_loss = val(model, val_dl, criterion)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

In [ ]:
epochs = range(1, num_epochs + 1)
plt.plot(epochs, train_losses, 'b', label='Training loss')
plt.plot(epochs, val_losses, 'r', label='Validation loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()